# 47 — FAISS Vector Search
**Goal:** Use FAISS for fast semantic similarity search over resume embeddings.

Chapter 46 computed one cosine similarity per resume–JD pair. That is fine for a handful of candidates, but an ATS may hold hundreds of thousands of resumes — brute-force comparison becomes the bottleneck. FAISS (Facebook AI Similarity Search) is a C++ library with a Python wrapper, built exactly for this: finding the nearest neighbors of a query vector in a large, high-dimensional collection in milliseconds.

**Why it matters for resumes / ATS:** "find the best 50 candidates for this JD" is a nearest-neighbor query over resume embeddings. FAISS turns it from an O(N) scan into an index lookup, and its approximate index types (IVF, HNSW) trade a little accuracy for orders of magnitude in speed — the standard trick for keeping talent search interactive at scale.

## 1. FAISS Basics

FAISS separates the **index** (the data structure holding all vectors) from the **search** (nearest-neighbor queries against that structure). Different index types sit on a speed-versus-accuracy spectrum:

| Index type | Search | Accuracy | Notes |
|---|---|---|---|
| `IndexFlatL2` | exact, brute force | best | L2 distance; O(N) per query |
| `IndexFlatIP` | exact, brute force | best | inner product; equals cosine after L2-normalizing |
| `IndexIVFFlat` | approximate | good | clusters vectors, probes only nearby clusters |
| `IndexHNSWFlat` | approximate | good–best | graph-based; fast and accurate, heavier memory |

**What the code does:** prints this design summary — FAISS's purpose, the index-type table, and the rule of thumb that `IndexFlatIP` is the right default below ~100K resumes because exact search is still fast enough.

In [ ]:
print('''FAISS = Facebook AI Similarity Search
Purpose: Fast nearest-neighbor search in high-dimensional spaces
Key: Build an index of resume embeddings, query with JD embedding

Index types:
- IndexFlatL2  -> exact L2 distance (brute force, most accurate)
- IndexFlatIP  -> inner product / cosine similarity
- IndexIVFFlat -> approximate (faster, slightly less accurate)
- IndexHNSWFlat -> graph-based (fast, good accuracy)

For <100K resumes: IndexFlatIP is fine (exact search).''')

## 2. Building a FAISS Index

Building an index is a two-step routine: pick the metric, then `add()` the vectors. The chapter uses `IndexFlatIP` (inner product) with **L2-normalized** vectors, which makes inner product equivalent to cosine similarity — the same metric Ch. 46 used for matching.

**What the code does:**
- Sets `d = 384`, the output dimension of `all-MiniLM-L6-v2`, and creates an empty `IndexFlatIP(d)`.
- Simulates 10 resume embeddings as `np.random.randn(10, d)` with a fixed seed, casts them to `float32` (FAISS requires this dtype), then normalizes each row with `faiss.normalize_L2`.
- Adds the matrix and prints `index.ntotal` and `index.d`.

**Expected:** the first print shows `ntotal = 0` with `dimension 384`; after `add()` it shows `ntotal = 10`. A real pipeline would use `model.encode()` (as in Ch. 46) instead of `randn` — the random vectors only demonstrate the mechanics.

In [ ]:
import numpy as np
import faiss

d = 384  # dimension (all-MiniLM-L6-v2)
index = faiss.IndexFlatIP(d)
print(f"Empty index: {index.ntotal} vectors, dimension {index.d}")

# Simulate resume embeddings
np.random.seed(42)
resume_embeddings = np.random.randn(10, d).astype(np.float32)
# Normalize for cosine similarity
faiss.normalize_L2(resume_embeddings)
index.add(resume_embeddings)
print(f"After adding 10 resumes: {index.ntotal} vectors")

## 3. Querying with a JD

With the index built, retrieval is one call: `index.search(query_vector, k)` returns the `k` nearest neighbors as two arrays — `scores` (the similarity values) and `indices` (which resume rows they belong to).

**What the code does:**
- Encodes the "JD" as a single random `(1, d)` vector, normalized exactly like the index contents.
- Searches with `k = 3` and prints each hit's rank, resume id, and score.

**Expected behavior:** you get 3 results ranked by descending inner product; because all vectors are unit-length, scores fall in the cosine range [-1, 1]. One caveat worth stating plainly: with random embeddings the "matches" carry no meaning — the point is the mechanics. Feeding real embeddings from Ch. 46's matcher would make the top hit the resume that genuinely talks about the same topics as the JD.

In [ ]:
# Query: job description embedding
jd_embedding = np.random.randn(1, d).astype(np.float32)
faiss.normalize_L2(jd_embedding)

# Search top-3 most similar resumes
k = 3
scores, indices = index.search(jd_embedding, k)
print("Top matches:")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    print(f"  Rank {i+1}: Resume {idx} (score: {score:.4f})")

## 4. Index Persistence

An index built in memory dies with the process. FAISS persists indices to disk with `write_index()` / `read_index()`, so a nightly re-embedding job can build once and serve queries all day.

**What the code does:**
- Saves the flat index to `/tmp/resume_index.faiss` and reloads it, verifying that `ntotal` and `d` survived the round trip.
- Builds a second, approximate index: `IndexIVFFlat(quantizer, d, nlist=2, METRIC_INNER_PRODUCT)`, which partitions the space into 2 clusters. IVF **must be trained** (`ivf_index.train()`) before `add()` — the quantizer needs data to learn cluster centroids.
- Sets `nprobe = 1`, meaning each query inspects only 1 of the 2 clusters.

**Expected:** the reloaded index reports the same 10 vectors; the IVF search returns similar-but-not-necessarily-identical results to the exact search, because probing one cluster skips vectors in the other. Higher `nprobe` = more accuracy, slower queries — the core knob of approximate search.

In [ ]:
faiss.write_index(index, "/tmp/resume_index.faiss")
loaded_index = faiss.read_index("/tmp/resume_index.faiss")
print(f"Saved and reloaded: {loaded_index.ntotal} vectors, d={loaded_index.d}")

# IVF for larger scale
nlist = 2
quantizer = faiss.IndexFlatIP(d)
ivf_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
ivf_index.train(resume_embeddings)
ivf_index.add(resume_embeddings)
ivf_index.nprobe = 1  # number of probes at search time
scores_ivf, indices_ivf = ivf_index.search(jd_embedding, k)
print(f"IVF results: indices={indices_ivf[0]}, scores={scores_ivf[0].round(4)}")

## Summary: FAISS enables fast resume retrieval from large candidate pools. IVF scales to millions.

FAISS indexes resume embeddings once, then answers "closest resumes to this JD" in milliseconds: `IndexFlatIP` gives exact cosine search for pools up to ~100K, and `IndexIVFFlat` (or HNSW) trades a little accuracy for the ability to scale to millions by clustering and probing only nearby regions. The index is persistable, so production systems build offline and serve online. What FAISS does *not* provide is metadata handling — filtering by role, seniority, or location requires separate bookkeeping or a database layer, which is exactly the gap Ch. 48 (ChromaDB) fills.

## Key Insight

**At scale, search speed is a product feature — FAISS is how you keep it interactive.**

For a candidate pool measured in thousands or millions, exact per-pair scoring (Ch. 46) stops being viable; nearest-neighbor indexes turn the same cosine similarity into a sub-linear lookup. The design lesson is the speed–accuracy trade-off: flat indexes are exact but O(N), IVF/HNSW are approximate but fast, and `nprobe` lets you dial between them per workload. This chapter's indexes store vectors only; Ch. 48 layers metadata and persistence on top, and Ch. 49 benchmarks whether the embeddings being searched are good.